# Análise de Fleiss' Kappa

Este notebook calcula o Fleiss' Kappa para medir concordância entre **três anotadores**:
- **Mendes** (anotador 1)
- **Guilherme** (anotador 2)
- **Marcos** (anotador 3)

Fleiss' Kappa é uma extensão do Cohen's Kappa para múltiplos anotadores, avaliando concordância além do acaso.
Escala: -1 (discordância total) a 1 (concordância perfeita).

In [44]:
# Importações necessárias
# =======================
# statsmodels: cálculos estatísticos (Fleiss' Kappa, aggregate_raters)
# pandas: manipulação de dados (leitura/escrita Excel)
# numpy: operações numéricas
# os: manipulação de caminhos do sistema de arquivos
# sys: informações do ambiente Python
# sklearn: métricas adicionais (confusion_matrix, classification_report)
# collections: Counter para contabilizar discordâncias

from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import pandas as pd
import numpy as np
import os
import sys
from collections import Counter

# Diagnóstico de versões
print("="*70)
print("DIAGNÓSTICO DE AMBIENTE - FLEISS' KAPPA")
print("="*70)
print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
try:
    import statsmodels as sm
    print(f"statsmodels version: {sm.__version__}")
except Exception:
    print("statsmodels: importação OK (versão não disponível)")
print("="*70)

DIAGNÓSTICO DE AMBIENTE - FLEISS' KAPPA
Python version: 3.12.6
pandas version: 3.0.2
numpy version: 2.4.4
statsmodels version: 0.14.6


In [45]:
# Configuração de caminhos e variáveis
# ====================================
# Define diretórios de entrada/saída para os arquivos Excel
# Todos os arquivos devem estar em: data/estatistica_kappa/

CAMINHO_DIRETORIO = os.path.abspath(os.path.join(
    os.path.dirname("sistema-inteligencia-prescritiva-para-industria-farmaceutica"), 
    "..", "..", "data", "estatistica_kappa"
))

# Caminho do arquivo de entrada (contém avaliações de todos os anotadores)
EXCEL_ENTRADA = os.path.join(CAMINHO_DIRETORIO, "noticias_classificadas.xlsx")

# Caminho do arquivo de saída com relatório de Fleiss' Kappa
EXCEL_SAIDA = os.path.join(CAMINHO_DIRETORIO, "fleiss_kappa_mendes_guilherme_marcos.xlsx")

# Nomes dos três anotadores
ANOTADORES = ['Mendes', 'Guilherme', 'Marcos']
COLUNAS_ANOTADORES = ['classificacao mendes', 'classificacao guilherme', 'classificacao marcos']

print("\n" + "="*70)
print("CONFIGURAÇÃO DE ARQUIVOS")
print("="*70)
print(f"Diretório de trabalho: {CAMINHO_DIRETORIO}")
print(f"Arquivo de entrada: {EXCEL_ENTRADA}")
print(f"Arquivo de saída: {EXCEL_SAIDA}")
print(f"Anotadores: {', '.join(ANOTADORES)}")
print(f"Arquivo de entrada existe: {os.path.exists(EXCEL_ENTRADA)}")
print("="*70)


CONFIGURAÇÃO DE ARQUIVOS
Diretório de trabalho: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa
Arquivo de entrada: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\noticias_classificadas.xlsx
Arquivo de saída: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\fleiss_kappa_mendes_guilherme_marcos.xlsx
Anotadores: Mendes, Guilherme, Marcos
Arquivo de entrada existe: True


In [46]:
# Carregamento e preparação dos dados
# ====================================
# Lê o arquivo Excel com as avaliações de todos os anotadores
# Extrai as colunas de interesse (Mendes, Guilherme, Marcos)
# Realiza limpeza e padronização dos dados

print("\nCarregando arquivo de entrada...")
try:
    df = pd.read_excel(EXCEL_ENTRADA)
    print(f"✓ Arquivo lido com sucesso")
    print(f"  Total de linhas: {len(df)}")
    print(f"  Total de colunas: {len(df.columns)}")
    print(f"  Colunas disponíveis: {list(df.columns)}")
except FileNotFoundError:
    print(f"✗ Erro: Arquivo não encontrado em {EXCEL_ENTRADA}")
    raise
except Exception as e:
    print(f"✗ Erro ao ler arquivo: {e}")
    raise

# Verifica se as colunas esperadas existem
print("\nValidando colunas esperadas...")
colunas_encontradas = [c for c in COLUNAS_ANOTADORES if c in df.columns]
colunas_faltantes = [c for c in COLUNAS_ANOTADORES if c not in df.columns]

if colunas_faltantes:
    print(f"✗ Colunas faltantes: {colunas_faltantes}")
    print("\nTentando encontrar colunas com nomes parcialmente iguais...")
    df_cols_normalized = {col: col.lower().strip() for col in df.columns}
    for col_norm, col_orig in df_cols_normalized.items():
        for anotador in ANOTADORES:
            if anotador.lower() in col_norm and 'classificacao' in col_norm:
                print(f"  Encontrado: '{col_orig}' (contém '{anotador}' e 'classificacao')")
else:
    print(f"✓ Todas as colunas encontradas: {colunas_encontradas}")

# Extrai as três colunas de avaliação
print("\nExtraindo dados dos anotadores...")
df_anotadores = df[COLUNAS_ANOTADORES].copy()

# Padroniza valores: uppercase, remove espaços
for col in COLUNAS_ANOTADORES:
    df_anotadores[col] = df_anotadores[col].astype(str).str.strip().str.upper()

for i, anotador in enumerate(ANOTADORES):
    n_valores = df_anotadores.iloc[:, i].count()
    print(f"  {anotador} - total de avaliações: {len(df_anotadores)} (valores válidos: {n_valores})")

# Remove linhas com qualquer valor vazio/NaN
mask_validas = df_anotadores.apply(lambda row: all(val not in ['', 'NAN', 'NONE', 'NAN'] for val in row), axis=1)
df_anotadores_validos = df_anotadores[mask_validas].reset_index(drop=True)

linhas_removidas = len(df_anotadores) - len(df_anotadores_validos)
print(f"\n  Linhas com valores válidos (para Fleiss' Kappa): {len(df_anotadores_validos)}")
print(f"  Linhas removidas (valores vazios/NaN): {linhas_removidas}")

# Exibe exemplos de dados
print("\nPrimeiras 5 avaliações dos três anotadores:")
print(df_anotadores_validos.head().to_string())

# Classes únicas identificadas
todas_classes_set = set()
for col in COLUNAS_ANOTADORES:
    todas_classes_set.update(df_anotadores_validos[col].unique())
todas_classes = sorted(todas_classes_set)

print(f"\nClasses únicas identificadas: {todas_classes}")
print(f"Total de classes: {len(todas_classes)}")


Carregando arquivo de entrada...
✓ Arquivo lido com sucesso
  Total de linhas: 1415
  Total de colunas: 17
  Colunas disponíveis: ['ano', 'mes', 'dia', 'palavra_chave', 'api_key_indice', 'titulo', 'descricao', 'conteudo', 'url', 'fonte_nome', 'fonte_url', 'classificacao mendes', 'classificacao guilherme', 'classificacao marcos', 'classificacao kappa', 'categoria_geral', 'subcategoria']

Validando colunas esperadas...
✓ Todas as colunas encontradas: ['classificacao mendes', 'classificacao guilherme', 'classificacao marcos']

Extraindo dados dos anotadores...
  Mendes - total de avaliações: 1415 (valores válidos: 1415)
  Guilherme - total de avaliações: 1415 (valores válidos: 1415)
  Marcos - total de avaliações: 1415 (valores válidos: 1415)

  Linhas com valores válidos (para Fleiss' Kappa): 1415
  Linhas removidas (valores vazios/NaN): 0

Primeiras 5 avaliações dos três anotadores:
  classificacao mendes classificacao guilherme classificacao marcos
0                MÉDIO              

In [47]:
# Preparação de dados para Fleiss' Kappa
# ======================================
# Fleiss' Kappa requer uma matriz NxM onde:
# N = número de itens (linhas/notícias)
# M = número de categorias
# Cada célula (i,j) contém a contagem de vezes que a categoria j foi atribuída ao item i

print("\n" + "="*70)
print("PREPARAÇÃO DE DADOS PARA FLEISS' KAPPA")
print("="*70)

print("\nTransformando avaliações em matriz de contagem...")
try:
    # aggregate_raters gera a matriz de contagem necessária para Fleiss' Kappa
    matriz_contagem, categorias_retornadas = aggregate_raters(df_anotadores_validos)
    
    print(f"✓ Matriz de contagem criada com sucesso")
    print(f"  Dimensões (itens x categorias): {matriz_contagem.shape}")
    print(f"  Número de itens: {matriz_contagem.shape[0]}")
    print(f"  Número de categorias: {matriz_contagem.shape[1]}")
    print(f"  Categorias (labels): {categorias_retornadas}")
    
    # Exemplo: primeiras 5 linhas da matriz
    print("\nExemplo (5 primeiras itens da matriz de contagem):")
    print("Formato: cada linha = contagens por categoria para um item")
    print(pd.DataFrame(matriz_contagem[:5], columns=categorias_retornadas).to_string())
    
except Exception as e:
    print(f"✗ Erro ao agregar avaliadores: {e}")
    raise


PREPARAÇÃO DE DADOS PARA FLEISS' KAPPA

Transformando avaliações em matriz de contagem...
✓ Matriz de contagem criada com sucesso
  Dimensões (itens x categorias): (1415, 3)
  Número de itens: 1415
  Número de categorias: 3
  Categorias (labels): ['ALTO' 'BAIXO' 'MÉDIO']

Exemplo (5 primeiras itens da matriz de contagem):
Formato: cada linha = contagens por categoria para um item
   ALTO  BAIXO  MÉDIO
0     0      2      1
1     1      1      1
2     0      2      1
3     1      1      1
4     2      0      1


In [48]:
# Cálculo do Fleiss' Kappa
# ========================
# Fleiss' Kappa mede concordância entre múltiplos anotadores
# Fórmula: κ = (P̄_o - P̄_e) / (1 - P̄_e)
# onde:
#   P̄_o = proporção média de acordo observado
#   P̄_e = proporção média de acordo esperado por acaso
# Interpretação:
#   κ < 0.20: Concordância fraca
#   0.20 ≤ κ < 0.40: Concordância razoável
#   0.40 ≤ κ < 0.60: Concordância moderada
#   0.60 ≤ κ < 0.80: Concordância substancial
#   κ ≥ 0.80: Concordância quase perfeita

print("\n" + "="*70)
print("CÁLCULO DO FLEISS' KAPPA")
print("="*70)

try:
    kappa = fleiss_kappa(matriz_contagem)
    print(f"\n✓ Fleiss' Kappa calculado com sucesso")
    print(f"  Kappa = {kappa:.4f}")
    
    # Interpretação do valor
    if kappa < 0.20:
        interpretacao = "Concordância FRACA"
    elif kappa < 0.40:
        interpretacao = "Concordância RAZOÁVEL"
    elif kappa < 0.60:
        interpretacao = "Concordância MODERADA"
    elif kappa < 0.80:
        interpretacao = "Concordância SUBSTANCIAL"
    else:
        interpretacao = "Concordância QUASE PERFEITA"
    
    print(f"  Interpretação: {interpretacao}")
    
    # Significado em termos práticos
    print(f"\n  Significado: Os {len(ANOTADORES)} anotadores têm {interpretacao.lower()}.")
    if kappa >= 0.60:
        print(f"  Recomendação: A concordância é aceitável para uso em análises.")
    elif kappa >= 0.40:
        print(f"  Recomendação: A concordância é moderada; considere revisar casos de desacordo.")
    else:
        print(f"  Recomendação: A concordância é baixa; revisar critérios e treinar anotadores.")
    
except Exception as e:
    print(f"✗ Erro ao calcular Fleiss' Kappa: {e}")
    raise


CÁLCULO DO FLEISS' KAPPA

✓ Fleiss' Kappa calculado com sucesso
  Kappa = 0.1858
  Interpretação: Concordância FRACA

  Significado: Os 3 anotadores têm concordância fraca.
  Recomendação: A concordância é baixa; revisar critérios e treinar anotadores.


In [49]:
# Análise de concordância por categoria
# ======================================
# Calcula a concordância para cada classe/categoria individualmente

print("\n" + "="*70)
print("ANÁLISE DE CONCORDÂNCIA POR CATEGORIA")
print("="*70)

print("\nConcordância por classe (proporção de itens onde todos concordaram):")
for j, categoria in enumerate(categorias_retornadas):
    # Calcula concordância por categoria usando método de Fleiss
    total_anotacoes_categoria = matriz_contagem[:, j].sum()
    itens_concordancia_perfeita = (matriz_contagem[:, j] == 3).sum()  # Todos os 3 anotadores
    itens_concordancia_parcial = (matriz_contagem[:, j] >= 2).sum()   # Pelo menos 2 anotadores
    
    percentual_perfeita = (itens_concordancia_perfeita / len(matriz_contagem)) * 100 if len(matriz_contagem) > 0 else 0
    percentual_parcial = (itens_concordancia_parcial / len(matriz_contagem)) * 100 if len(matriz_contagem) > 0 else 0
    
    print(f"\n  {categoria}:")
    print(f"    Total de anotações: {total_anotacoes_categoria}")
    print(f"    Concordância perfeita (3/3): {itens_concordancia_perfeita} itens ({percentual_perfeita:.1f}%)")
    print(f"    Concordância parcial (≥2/3): {itens_concordancia_parcial} itens ({percentual_parcial:.1f}%)")


ANÁLISE DE CONCORDÂNCIA POR CATEGORIA

Concordância por classe (proporção de itens onde todos concordaram):

  ALTO:
    Total de anotações: 895
    Concordância perfeita (3/3): 47 itens (3.3%)
    Concordância parcial (≥2/3): 247 itens (17.5%)

  BAIXO:
    Total de anotações: 1528
    Concordância perfeita (3/3): 154 itens (10.9%)
    Concordância parcial (≥2/3): 493 itens (34.8%)

  MÉDIO:
    Total de anotações: 1822
    Concordância perfeita (3/3): 187 itens (13.2%)
    Concordância parcial (≥2/3): 511 itens (36.1%)


In [50]:
# Análise de padrões de concordância
# ===================================
# Identifica os tipos de concordância mais comuns entre os anotadores

print("\n" + "="*70)
print("ANÁLISE DE PADRÕES DE CONCORDÂNCIA")
print("="*70)

# Calcula concordância entre pares
print("\nConcordância entre pares de anotadores:")

pares = [
    (0, 1, 'Mendes vs Guilherme'),
    (0, 2, 'Mendes vs Marcos'),
    (1, 2, 'Guilherme vs Marcos')
]

concordancia_pares = {}
for i, j, nome_par in pares:
    col_i = COLUNAS_ANOTADORES[i]
    col_j = COLUNAS_ANOTADORES[j]
    
    acordos = (df_anotadores_validos[col_i] == df_anotadores_validos[col_j]).sum()
    total = len(df_anotadores_validos)
    percentual = (acordos / total) * 100 if total > 0 else 0
    
    concordancia_pares[nome_par] = (acordos, total, percentual)
    print(f"  {nome_par}: {acordos} acordos de {total} ({percentual:.1f}%)")

# Concordância entre os 3 anotadores
print("\nConcordância entre os 3 anotadores:")
todos_concordam = (df_anotadores_validos.iloc[:, 0] == 
                   df_anotadores_validos.iloc[:, 1]).values & \
                  (df_anotadores_validos.iloc[:, 1] == 
                   df_anotadores_validos.iloc[:, 2]).values
total_concordancia_3 = todos_concordam.sum()
percentual_concordancia_3 = (total_concordancia_3 / len(df_anotadores_validos)) * 100

print(f"  Todos os 3 concordam: {total_concordancia_3} de {len(df_anotadores_validos)} ({percentual_concordancia_3:.1f}%)")


ANÁLISE DE PADRÕES DE CONCORDÂNCIA

Concordância entre pares de anotadores:
  Mendes vs Guilherme: 560 acordos de 1415 (39.6%)
  Mendes vs Marcos: 729 acordos de 1415 (51.5%)
  Guilherme vs Marcos: 738 acordos de 1415 (52.2%)

Concordância entre os 3 anotadores:
  Todos os 3 concordam: 388 de 1415 (27.4%)


In [51]:
# Distribuição de classificações por anotador
# ============================================
# Mostra quantas vezes cada classe foi atribuída por cada anotador

print("\n" + "="*70)
print("DISTRIBUIÇÃO DE CLASSIFICAÇÕES POR ANOTADOR")
print("="*70)

for i, anotador in enumerate(ANOTADORES):
    print(f"\nAvaliações de {anotador}:")
    col = COLUNAS_ANOTADORES[i]
    contagem = df_anotadores_validos[col].value_counts().sort_index()
    
    for classe, count in contagem.items():
        percentual = (count / len(df_anotadores_validos)) * 100
        barra = '█' * int(percentual / 2)  # Barra visual
        print(f"  {classe}: {count:4d} ({percentual:5.1f}%) {barra}")
    print(f"  Total: {contagem.sum()}")

# Comparação lado a lado
print("\nComparação de distribuições (Mendes vs Guilherme vs Marcos):")
comparacao_dist = pd.DataFrame()
for i, anotador in enumerate(ANOTADORES):
    col = COLUNAS_ANOTADORES[i]
    contagens_por_classe = {}
    for classe in todas_classes:
        n = (df_anotadores_validos[col] == classe).sum()
        pct = (n / len(df_anotadores_validos)) * 100
        contagens_por_classe[classe] = f"{n} ({pct:.1f}%)"
    comparacao_dist[anotador] = contagens_por_classe

print(comparacao_dist.fillna('0 (0.0%)').to_string())


DISTRIBUIÇÃO DE CLASSIFICAÇÕES POR ANOTADOR

Avaliações de Mendes:
  ALTO:  208 ( 14.7%) ███████
  BAIXO:  487 ( 34.4%) █████████████████
  MÉDIO:  720 ( 50.9%) █████████████████████████
  Total: 1415

Avaliações de Guilherme:
  ALTO:  451 ( 31.9%) ███████████████
  BAIXO:  361 ( 25.5%) ████████████
  MÉDIO:  603 ( 42.6%) █████████████████████
  Total: 1415

Avaliações de Marcos:
  ALTO:  236 ( 16.7%) ████████
  BAIXO:  680 ( 48.1%) ████████████████████████
  MÉDIO:  499 ( 35.3%) █████████████████
  Total: 1415

Comparação de distribuições (Mendes vs Guilherme vs Marcos):
            Mendes    Guilherme       Marcos
ALTO   208 (14.7%)  451 (31.9%)  236 (16.7%)
BAIXO  487 (34.4%)  361 (25.5%)  680 (48.1%)
MÉDIO  720 (50.9%)  603 (42.6%)  499 (35.3%)


In [53]:
# Identificação de itens problemáticos (com baixa concordância)
# ==============================================================
# Encontra itens onde os anotadores discordam

print("\n" + "="*70)
print("ANÁLISE DE ITENS COM DISCORDÂNCIA")
print("="*70)

# Calcula discordância para cada item
discordancia_por_item = []
for idx in range(len(df_anotadores_validos)):
    avaliacoes = df_anotadores_validos.iloc[idx].values
    # Se todos são iguais, discordância = 0; caso contrário discordância = 1
    tem_discordancia = len(set(avaliacoes)) > 1
    discordancia_por_item.append(tem_discordancia)

total_itens_discordancia = sum(discordancia_por_item)
percentual_discordancia = (total_itens_discordancia / len(df_anotadores_validos)) * 100

print(f"\nTotal de itens com discordância: {total_itens_discordancia} de {len(df_anotadores_validos)} ({percentual_discordancia:.1f}%)")
print(f"Total de itens com concordância: {len(df_anotadores_validos) - total_itens_discordancia} ({100-percentual_discordancia:.1f}%)")

# Tipos de discordância mais comuns
print("\nPadrões de discordância mais comuns (top 10):")
padroes_discordancia = []
for idx in range(len(df_anotadores_validos)):
    avaliacoes = sorted(df_anotadores_validos.iloc[idx].values)
    if len(set(avaliacoes)) > 1:  # Se há discordância
        padroes_discordancia.append(tuple(avaliacoes))

if padroes_discordancia:
    padroes_contagem = Counter(padroes_discordancia)
    for (padrao), count in padroes_contagem.most_common(10):
        percentual_padrao = (count / total_itens_discordancia) * 100
        print(f"  {padrao}: {count} vezes ({percentual_padrao:.1f}% dos desacordos)")
else:
    print("  Sem discordâncias (concordância perfeita!)")


ANÁLISE DE ITENS COM DISCORDÂNCIA

Total de itens com discordância: 1027 de 1415 (72.6%)
Total de itens com concordância: 388 (27.4%)

Padrões de discordância mais comuns (top 10):
  ('BAIXO', 'BAIXO', 'MÉDIO'): 266 vezes (25.9% dos desacordos)
  ('BAIXO', 'MÉDIO', 'MÉDIO'): 207 vezes (20.2% dos desacordos)
  ('ALTO', 'ALTO', 'MÉDIO'): 183 vezes (17.8% dos desacordos)
  ('ALTO', 'BAIXO', 'MÉDIO'): 164 vezes (16.0% dos desacordos)
  ('ALTO', 'MÉDIO', 'MÉDIO'): 117 vezes (11.4% dos desacordos)
  ('ALTO', 'BAIXO', 'BAIXO'): 73 vezes (7.1% dos desacordos)
  ('ALTO', 'ALTO', 'BAIXO'): 17 vezes (1.7% dos desacordos)
  ('BAIXO', 'BAIXO', 'MÉDIO'): 266 vezes (25.9% dos desacordos)
  ('BAIXO', 'MÉDIO', 'MÉDIO'): 207 vezes (20.2% dos desacordos)
  ('ALTO', 'ALTO', 'MÉDIO'): 183 vezes (17.8% dos desacordos)
  ('ALTO', 'BAIXO', 'MÉDIO'): 164 vezes (16.0% dos desacordos)
  ('ALTO', 'MÉDIO', 'MÉDIO'): 117 vezes (11.4% dos desacordos)
  ('ALTO', 'BAIXO', 'BAIXO'): 73 vezes (7.1% dos desacordos)
  ('

In [54]:
# Salvamento de relatório em arquivo Excel
# ==========================================
# Cria um arquivo com análise completa da concordância

print("\n" + "="*70)
print("SALVAMENTO DE RELATÓRIO")
print("="*70)

try:
    with pd.ExcelWriter(EXCEL_SAIDA, engine='openpyxl') as writer:
        # Aba 1: Resumo Executivo
        resumo_data = {
            'Métrica': [
                "Fleiss' Kappa",
                'Interpretação',
                'Total de Itens Analisados',
                'Itens com Concordância Perfeita (3/3)',
                'Itens com Discordância',
                'Porcentagem de Concordância Perfeita (%)',
                'Porcentagem de Discordância (%)',
                'Número de Classes/Categorias',
                'Anotadores'
            ],
            'Valor': [
                f"{kappa:.4f}",
                interpretacao,
                len(df_anotadores_validos),
                total_concordancia_3,
                total_itens_discordancia,
                f"{percentual_concordancia_3:.2f}",
                f"{percentual_discordancia:.2f}",
                len(todas_classes),
                ', '.join(ANOTADORES)
            ]
        }
        resumo_df = pd.DataFrame(resumo_data)
        resumo_df.to_excel(writer, sheet_name='Resumo Executivo', index=False)
        print(f"✓ Aba 'Resumo Executivo' criada")
        
        # Aba 2: Concordância entre Pares
        pares_data = []
        for nome_par, (acordos, total, pct) in concordancia_pares.items():
            pares_data.append({
                'Par de Anotadores': nome_par,
                'Acordos': acordos,
                'Total de Itens': total,
                'Porcentagem (%)': f"{pct:.2f}"
            })
        pares_df = pd.DataFrame(pares_data)
        pares_df.to_excel(writer, sheet_name='Concordancia Pares', index=False)
        print(f"✓ Aba 'Concordancia Pares' criada")
        
        # Aba 3: Distribuição de Classificações
        dist_data = []
        for classe in todas_classes:
            linha = {'Classe': classe}
            for i, anotador in enumerate(ANOTADORES):
                col = COLUNAS_ANOTADORES[i]
                count = (df_anotadores_validos[col] == classe).sum()
                pct = (count / len(df_anotadores_validos)) * 100
                linha[f"{anotador} (count)"] = count
                linha[f"{anotador} (%)"] = f"{pct:.2f}"
            dist_data.append(linha)
        dist_df = pd.DataFrame(dist_data)
        dist_df.to_excel(writer, sheet_name='Distribuicao', index=False)
        print(f"✓ Aba 'Distribuicao' criada")
        
        # Aba 4: Concordância por Categoria
        categ_data = []
        for j, categoria in enumerate(categorias_retornadas):
            total_anotacoes = matriz_contagem[:, j].sum()
            perfeita = (matriz_contagem[:, j] == 3).sum()
            parcial = (matriz_contagem[:, j] >= 2).sum()
            pct_perfeita = (perfeita / len(matriz_contagem)) * 100 if len(matriz_contagem) > 0 else 0
            pct_parcial = (parcial / len(matriz_contagem)) * 100 if len(matriz_contagem) > 0 else 0
            
            categ_data.append({
                'Categoria': categoria,
                'Total Anotacoes': total_anotacoes,
                'Concordancia Perfeita (3/3)': perfeita,
                'Concordancia Perfeita (%)': f"{pct_perfeita:.2f}",
                'Concordancia Parcial (>=2/3)': parcial,
                'Concordancia Parcial (%)': f"{pct_parcial:.2f}"
            })
        categ_df = pd.DataFrame(categ_data)
        categ_df.to_excel(writer, sheet_name='Concordancia Categorias', index=False)
        print(f"✓ Aba 'Concordancia Categorias' criada")
        
        # Aba 5: Padrões de Discordância
        if padroes_discordancia:
            padroes_data = []
            padroes_contagem = Counter(padroes_discordancia)
            for padrao, count in padroes_contagem.most_common():
                pct = (count / total_itens_discordancia) * 100
                padroes_data.append({
                    'Padrão': f"{padrao[0]} | {padrao[1]} | {padrao[2]}",
                    'Frequência': count,
                    'Porcentagem (%)': f"{pct:.2f}"
                })
            padroes_df = pd.DataFrame(padroes_data)
            padroes_df.to_excel(writer, sheet_name='Padroes Discordancia', index=False)
            print(f"✓ Aba 'Padroes Discordancia' criada")
    
    print(f"\n✓ Arquivo salvo com sucesso em:")
    print(f"  {EXCEL_SAIDA}")
    
except Exception as e:
    print(f"✗ Erro ao salvar arquivo: {e}")
    raise


SALVAMENTO DE RELATÓRIO
✓ Aba 'Resumo Executivo' criada
✓ Aba 'Concordancia Pares' criada
✓ Aba 'Distribuicao' criada
✓ Aba 'Concordancia Categorias' criada
✓ Aba 'Padroes Discordancia' criada

✓ Arquivo salvo com sucesso em:
  c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\fleiss_kappa_mendes_guilherme_marcos.xlsx


In [58]:
# Resumo final da análise
# =======================

print("\n" + "="*70)
print("RESUMO FINAL DA ANÁLISE - FLEISS' KAPPA")
print("="*70)

print(f"\n✓ Análise concluída com sucesso!")
print(f"\nPrincipais Resultados:")
print(f"  Fleiss' Kappa (Mendes, Guilherme, Marcos): {kappa:.4f}")
print(f"  % de Concordância Fleiss' Kappa (Mendes, Guilherme, Marcos): {kappa*100:.2f}%")
print(f"  Interpretação: {interpretacao}")
print(f"\n  Total de itens analisados: {len(df_anotadores_validos)}")
print(f"  Concordância perfeita (3/3): {total_concordancia_3} ({percentual_concordancia_3:.1f}%)")
print(f"  Discordâncias: {total_itens_discordancia} ({percentual_discordancia:.1f}%)")
#print(f"\n  Arquivo de saída: {os.path.basename(EXCEL_SAIDA)}")
#print(f"  Localização: {EXCEL_SAIDA}")
print(f"\nReferência de Interpretação:")
print(f"  κ < 0.20: Concordância FRACA")
print(f"  0.20-0.40: Concordância RAZOÁVEL")
print(f"  0.40-0.60: Concordância MODERADA")
print(f"  0.60-0.80: Concordância SUBSTANCIAL")
print(f"  κ ≥ 0.80: Concordância QUASE PERFEITA")
print("="*70)


RESUMO FINAL DA ANÁLISE - FLEISS' KAPPA

✓ Análise concluída com sucesso!

Principais Resultados:
  Fleiss' Kappa (Mendes, Guilherme, Marcos): 0.1858
  % de Concordância Fleiss' Kappa (Mendes, Guilherme, Marcos): 18.58%
  Interpretação: Concordância FRACA

  Total de itens analisados: 1415
  Concordância perfeita (3/3): 388 (27.4%)
  Discordâncias: 1027 (72.6%)

Referência de Interpretação:
  κ < 0.20: Concordância FRACA
  0.20-0.40: Concordância RAZOÁVEL
  0.40-0.60: Concordância MODERADA
  0.60-0.80: Concordância SUBSTANCIAL
  κ ≥ 0.80: Concordância QUASE PERFEITA


In [36]:
# Estatísticas de concordância - Fleiss' Kappa
# Este notebook calcula o Kappa de Fleiss a partir de avaliações de 3 anotadores.
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import pandas as pd
import os
import sys
import requests
from dotenv import load_dotenv
load_dotenv()

DIAGNÓSTICO DE AMBIENTE - FLEISS' KAPPA ANALYSIS
Python version: 3.12.6
pandas version: 3.0.2
numpy version: 2.4.4
statsmodels version: 0.14.6
scikit-learn version: 1.8.0


In [37]:

# CAMINHO DOS ARQUIVOS
CAMINHO_DIRETORIO = os.path.abspath(os.path.join(os.path.dirname("sistema-inteligencia-prescritiva-para-industria-farmaceutica"), "..","..", "data","estatistica_kappa"))
EXCEL_ENTRADA = os.path.join(CAMINHO_DIRETORIO, "noticias_classificadas.xlsx")


CONFIGURAÇÃO DE ARQUIVOS
Diretório de trabalho: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa
Arquivo de entrada: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\noticias_classificadas.xlsx
Arquivo de entrada existe: True
Arquivo de saída (desempatadas): c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\avaliacoes_desempatadas.xlsx
Arquivo de relatório: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\fleiss_kappa_relatorio.xlsx


In [38]:
# Leitura do arquivo de entrada
try:
    df = pd.read_excel(EXCEL_ENTRADA)
    print(f"Arquivo lido com sucesso. Linhas: {len(df)}, colunas: {len(df.columns)}")
    print("Colunas encontradas:", list(df.columns))
except FileNotFoundError as e:
    print(f"Arquivo não encontrado: {EXCEL_ENTRADA}")
    raise

# Seleciona somente as colunas de classificação utilizadas para o Fleiss' Kappa
expected_cols = ['classificacao mendes','classificacao gilherme']#,'classificacao marcos']
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    print(f"Aviso: colunas esperadas ausentes: {missing}. Verifique o cabeçalho do Excel.")

# Se as colunas existem, cria df_real; caso contrário cria uma cópia com as colunas disponíveis (para mensagens de debug)
if not missing:
    df_real = df[expected_cols].copy()
else:
    # tenta localizar colunas parecidas (lowercased)
    cols_lower = [c.lower().strip() for c in df.columns]
    mapping = {}
    for c in expected_cols:
        if c in cols_lower:
            idx = cols_lower.index(c)
            mapping[c] = df.columns[idx]
    if mapping:
        print(f"Mapeamento de colunas encontrado: {mapping}")
        df_real = df[[mapping.get(c, mapping.get(c.lower(), None)) for c in expected_cols if mapping.get(c) or mapping.get(c.lower())]].copy()
    else:
        # cria um df vazio com as colunas esperadas para evitar erros posteriores
        df_real = pd.DataFrame(columns=expected_cols)

print("Exemplo (primeiras linhas) das colunas usadas para o Fleiss' Kappa:")
print(df_real.head())


CARREGAMENTO DE DADOS
Carregando arquivo de entrada...
✓ Arquivo lido com sucesso
  Total de linhas: 1415
  Total de colunas: 17
  Colunas disponíveis: ['ano', 'mes', 'dia', 'palavra_chave', 'api_key_indice', 'titulo', 'descricao', 'conteudo', 'url', 'fonte_nome', 'fonte_url', 'classificacao mendes', 'classificacao guilherme', 'classificacao marcos', 'classificacao kappa', 'categoria_geral', 'subcategoria']

Validando colunas esperadas...
✓ Todas as colunas encontradas: ['classificacao mendes', 'classificacao guilherme', 'classificacao marcos']

Extraindo dados dos anotadores...
  Mendes - total de avaliações: 1415
  Guilherme - total de avaliações: 1415
  Marcos - total de avaliações: 1415
  Linhas com valores válidos: 1415
  Linhas removidas (valores vazios/NaN): 0

Primeiras 5 avaliações:
  Mendes Guilherme Marcos
0  MÉDIO     BAIXO  BAIXO
1   ALTO     MÉDIO  BAIXO
2  MÉDIO     BAIXO  BAIXO
3   ALTO     MÉDIO  BAIXO
4   ALTO      ALTO  MÉDIO

Classes identificadas por Mendes: ['ALT

In [43]:
# 3. Transforma em matriz de contagem e 4. Calcula o Kappa
print("Transformando avaliações em matriz de contagem para o Fleiss' Kappa...")
try:
    contagem, categorias = aggregate_raters(df_real)
    print(f"Matriz de contagem criada. shape: {contagem.shape}")
    print(f"Categorias (labels) identificadas: {categorias}")
except Exception as e:
    print(f"Erro ao agregar avaliadores: {e}")
    raise

try:
    kappa = fleiss_kappa(contagem)
    print(f"Kappa de Fleiss: {kappa:.4f}")
except Exception as e:
    print(f"Erro ao calcular o Kappa de Fleiss: {e}")
    raise
    

Transformando avaliações em matriz de contagem para o Fleiss' Kappa...
Erro ao agregar avaliadores: '<' not supported between instances of 'str' and 'int'


TypeError: '<' not supported between instances of 'str' and 'int'

In [40]:
# TRATAMENTO E DESEMPATE
# Cria/normaliza coluna de saída
df['classificacao kappa'] = df['classificacao kappa'].fillna('').astype(str)

cont_acordo = 0
cont_divergencia = 0

# PADRONIZAÇÃO: transforma nomes de colunas em lowercase (ajusta referências a sdf_realuir)
df.codf_realmns = df.columns.str.lower().str.strip()
print(f"Iniciando loop de desempate df_real {len(df)} linhas...")

for index, row in df.iterrows():
    aval_mendes = str(row.get('classificacao mendes', '')).strip().upper()
    aval_guilherme = str(row.get('classificacao guilherme', '')).strip().upper()
    aval_marcos = str(row.get('classificacao marcos', '')).strip().upper()

    # CONDIÇÕES PARA DETERMINAR O DESEMPATE OU ACORDO
    if aval_mendes == aval_guilhermeval_mendesdf_real= '':
        df.at[index, 'classificacao kappa'] = aval_mendes
        print(f"Linha {index}: Acordo entre todos -> {aval_mendes}")
        cont_acordo += 1

    elif aval_mendes == aval_marcos and aval_guiland aval_men_realdes != '':
        df.at[index, 'classificacao kappa'] = aval_mendes
        print(f"Linha {index}: Acordo Mendes-Marcos -> {aval_mendes}")
        cont_acordo += 1

    elif aval_guilherme == aval_marcos and avae and aval_g_realuilhe rme != '':
        df.at[index, 'classificacao kappa'] = aval_guilherme
        print(f"Linha {index}: Acordo Guilherme-Marcos -> {aval_guilherme}")
        cont_acordo += 1

    elif aval_mendes == aval_guilhermeadal_mendes an_reald aval_mendes != '':
        df.at[index, 'classificacao kappa'] = aval_mendes
        print(f"Linha {index}: Acordo Mendes-Guilherme -> {aval_mendes}")
        cont_acordo += 1

    else:
        # Divergência entre os avaliadores: marca        df_real.at[index, 'classificacao kappa'] = ''.at[index, 'classificacao kappa'] = ''
        print(f"Linha {index}: Divergência ({aval_mendes} | {aval_guilherme} | {aval_marcos}).")
        cont_divergencia += 1

    # Progresso a cada 100 linhas para não poluir muito a saída
    if (index + 1) % 100 == 0:
    
         
        (f"Processadas {index+1} linhas...")

print(f"Total de Divergências: {cont_divergencia}")
print(f"Total de Acordos: {cont_acordo}")


PROCESSAMENTO DE DESEMPATE
Iniciando loop de desempate em 1415 linhas...
----------------------------------------------------------------------
Linha 0: Acordo Guilherme-Marcos → BAIXO (Mendes: MÉDIO)
Linha 1: DIVERGÊNCIA (ALTO vs MÉDIO vs BAIXO)
Linha 2: Acordo Guilherme-Marcos → BAIXO (Mendes: MÉDIO)
Linha 3: DIVERGÊNCIA (ALTO vs MÉDIO vs BAIXO)
Linha 4: Acordo Mendes-Guilherme → ALTO (Marcos: MÉDIO)
Linha 5: Acordo Mendes-Marcos → BAIXO (Guilherme: MÉDIO)
Linha 6: Acordo TODOS (ALTO)
Linha 7: Acordo TODOS (BAIXO)
Linha 8: Acordo TODOS (ALTO)
Linha 9: Acordo Guilherme-Marcos → ALTO (Mendes: MÉDIO)
Linha 10: Acordo Mendes-Marcos → BAIXO (Guilherme: MÉDIO)
Linha 11: Acordo Guilherme-Marcos → BAIXO (Mendes: MÉDIO)
Linha 12: Acordo Mendes-Marcos → BAIXO (Guilherme: MÉDIO)
Linha 13: Acordo Guilherme-Marcos → ALTO (Mendes: MÉDIO)
Linha 14: Acordo Mendes-Guilherme → MÉDIO (Marcos: ALTO)
Linha 15: Acordo Mendes-Guilherme → BAIXO (Marcos: MÉDIO)
Linha 16: Acordo Guilherme-Marcos → MÉDIO (Men


DISTRIBUIÇÃO DE CLASSIFICAÇÕES



DISTRIBUIÇÃO DE CLASSIFICAÇÕES


AttributeError: 'str' object has no attribute 'value_counts'